# Question 6 - DDPM Forward Process (closed form)
$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \qquad \epsilon\sim\mathcal{N}(0,I)$$

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
def add_noise(x_0, alpha_bar_t):
    eps = tf.random.normal(tf.shape(x_0))
    x_t = tf.sqrt(alpha_bar_t) * x_0 + tf.sqrt(1.0 - alpha_bar_t) * eps
    return x_t, eps

In [ ]:
# Demo on MNIST with a linear beta schedule, T = 1000
(x_train, _), _ = keras.datasets.mnist.load_data()
x0 = (x_train[:1].astype("float32") / 127.5 - 1.0)[..., None]     # scale to [-1, 1]

T = 1000
betas = np.linspace(1e-4, 0.02, T, dtype=np.float32)
alpha_bar = np.cumprod(1 - betas)

steps = [0, 50, 100, 250, 500, 999]
plt.figure(figsize=(12, 2))
for i, t in enumerate(steps):
    x_t, _ = add_noise(x0, alpha_bar[t])
    plt.subplot(1, len(steps), i + 1)
    plt.imshow(x_t[0, :, :, 0], cmap='gray'); plt.title(f"t={t}"); plt.axis('off')
plt.show()

### 2. Why is the closed form better for training?
- You can jump **directly to any random timestep t in one operation (O(1))** instead of running t sequential noising steps (O(t)).
- Training samples a random t per image in every batch, so this makes each step cheap and fully parallel on the GPU.
- It gives the exact same distribution $q(x_t|x_0)$ as step-by-step noising, and we also get the exact noise $\epsilon$ that the model must predict.